In [ ]:
import pandas as pd
import json
import numpy as np
import math
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
import torch.nn.functional as F
import matplotlib.pyplot as plt
import selfies as sf
from rdkit import Chem
from rdkit.Chem import Draw, AllChem, DataStructs, rdmolops, Descriptors, rdMolDescriptors, QED
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from tqdm import tqdm
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import r2_score
from scipy.signal import find_peaks

import sys
from pathlib import Path
molecula_path = './projects/Smiles-latent-project/ProbeVAE/models'
sys.path.append(molecula_path)
from SlotModelAR import VaeTransformer

spectra_path = '/projects/Spectras-latent-space/notebooks/'

In [2]:
data_dir = Path('/home/andrze06/projects/Spectras-latent-space/data/multimodal_spectroscopic_dataset')
files = sorted(data_dir.glob("aligned_chunk_*.parquet"))
dfs = [pd.read_parquet(f, columns=['smiles', 'c_nmr_spectra']) for f in files]
df = pd.concat(dfs, ignore_index=True)

In [3]:
df["selfies"] = [sf.encoder(sf.decoder(sf.encoder(s))) for s in tqdm(df["smiles"].tolist())]

100%|████████████████████████████████████████████████| 794386/794386 [12:36<00:00, 1050.54it/s]


In [4]:
df.head()

,smiles,c_nmr_spectra,selfies
0,COc1nc2ccccc2cc1C(=O)O,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[C][O][C][=N][C][=C][C][=C][C][=C][Ring1][=Bra...
1,CCOC(=O)c1cc2c(OCc3coc4cc(F)ccc34)cccc2n1C(=O)...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[C][C][O][C][=Branch1][C][=O][C][=C][C][=C][Br...
2,CCCCOc1c(CN2C(=O)c3ccccc3C2=O)n(CC2CC2)c(=O)c2...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[C][C][C][C][O][C][=C][Branch2][Ring1][Ring1][...
3,CCCCNC[C@@H]1O[C@](O)(CO)[C@@H](O)[C@@H]1O,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[C][C][C][C][N][C][C@@H1][O][C@][Branch1][C][O...
4,O=C(NCC1CC1)c1nc2c(N3CCC(n4c(=O)[nH]c5ccccc54)...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[O][=C][Branch1][Branch2][N][C][C][C][C][Ring1...


In [5]:
df.to_parquet(path='/home/andrze06/projects/Spectras-latent-space/data/spectra_selfies_df.parquet', index=False, compression='zstd')

In [ ]:
# data prep
df['tokens'] = df['selfies'].apply(lambda x: list(sf.split_selfies(x)))
vocab = sorted(set([tok for seq in df['tokens'] for tok in seq]))
PAD, SOS, EOS = "<PAD>", "<SOS>", "<EOS>"
vocab = [PAD, SOS, EOS] + vocab
vocab_size = len(vocab)

tok2id = {tok: idx for idx, tok in enumerate(vocab)}
id2tok = {idx: tok for idx, tok in enumerate(vocab)}

def molecule_tok2id(tokens, tok2id):
    return np.array([1] + [tok2id[t] for t in tokens] + [2])

df['token_ids'] = df['tokens'].apply(lambda x: molecule_tok2id(x, tok2id))
sequences = df['token_ids'].tolist()
max_len = max(len(seq) for seq in sequences)

# Spectra data
#spectra_data = np.array(df['c_nmr_spectra'].to_list(), dtype=np.float32)
scale = 1e4

def transform_data(targets_list):
    return [np.log1p(t * scale).astype(np.float32) for t in tqdm(targets_list)]

def inverse_transform_data(y):
    return np.expm1(y) / scale

class SpectraSmilesDataset(Dataset):
    def __init__(self, smiles, spectras, indices):
        self.smiles = smiles
        self.spectras = spectras
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        idx = self.indices[index]
        tokens = torch.tensor(self.smiles[idx], dtype=torch.long)
        spectra = torch.tensor(self.spectras[idx], dtype=torch.float32)
        spectra = torch.log1p(spectra * scale).float()
        return tokens, spectra

In [8]:
idx = np.arange(len(df))
train_idx, dummy_idx = train_test_split(idx, test_size=0.2, random_state=42)
val_idx, test_idx = train_test_split(dummy_idx, test_size=0.5, random_state=42)
    
train_data = SpectraSmilesDataset(sequences, df['c_nmr_spectra'].to_list(), train_idx)
val_data = SpectraSmilesDataset(sequences, df['c_nmr_spectra'].to_list(), val_idx)
test_data = SpectraSmilesDataset(sequences, df['c_nmr_spectra'].to_list(), test_idx)

In [9]:
# utils
@torch.no_grad()
def accuracy(model, loader, mode='eval', pad_id=0, device='cuda'):
    model.eval()
    total_correct = 0
    total_tok = 0
    total_seq = 0
    perfect = 0

    for x in loader:
        x = x.to(device)
        if mode == 'train':
            logits, _, _ = model(x, mode=mode)
            targets = x[:, 1:]
            pred = logits.argmax(dim=-1)
            mask = (targets != pad_id)
            correct = (pred == targets) & mask

            total_correct += correct.sum().item()
            total_tok += mask.sum().item()
            seq_correct = (correct.sum(dim=1) == mask.sum(dim=1))
            perfect += seq_correct.sum().item()
            total_seq += x.size(0)
        else:
            tokens, _, _ = model(x, mode='eval')

            for i, pred in enumerate(tokens):
                true = x[i]
                mask = (true != pad_id)
                true_len = mask.sum().item()
                if true_len == 0:
                    continue
                if len(pred) < true_len:
                    pad = torch.full((true_len - len(pred),), pad_id, device=pred.device, dtype=pred.dtype)
                    pred = torch.cat([pred, pad], dim=0)
                else:
                    pred = pred[:true_len]
                true = true[:true_len]
                correct = (pred == true)
                total_correct += correct.sum().item()
                total_tok += true_len
                perfect += int(correct.all())
                total_seq += 1
                
    return (total_correct / max(total_tok, 1), perfect / max(total_seq, 1))
     
def collate_fn(batch):
    tokens, spectra = zip(*batch)
    padded_tokens = torch.full((len(tokens), max_len), fill_value=tok2id[PAD], dtype=torch.long)

    for i, seq in enumerate(tokens):
        padded_tokens[i, :len(seq)] = seq

    spectra = torch.stack(spectra)
    return padded_tokens, spectra